# 6 · TruthfulQA — proper truthfulness judge (fix the broken BLEURT label)

On TruthfulQA the `BLEURT(answer, correct_refs) > 0.5` label is **invalid**: answers are free-form and adversarial, so they pile up at the 0.5 cut and get mislabelled (correct *'rainbows have no taste'* → halluc; wrong *'marry your grandparent'* → truthful). That's why every detector scored ≈0.5 or below in notebook 5 — they were graded against a broken key.

TruthfulQA ships **both `correct_answers` AND `incorrect_answers`**, so the right test is **comparative**: is the answer closer to a CORRECT reference than to an INCORRECT one? Two free, locally-cached judges do this:
- **`nli`** — a strong entailment model (DeBERTa-v3 MNLI, ~1 GB): does the answer *entail* a correct reference more than any incorrect one? (the 'strong model' check, tiny VRAM)
- **`bleurtacc`** — TruthfulQA's own automatic metric: max BLEURT to correct > max BLEURT to incorrect. (free, reuses the BLEURT env, no new model)

The detector **scores** already live in `data/truthfulqa_cross_eval.parquet` (from notebook 5) and don't depend on the label, so this just **swaps the label and recomputes AUROC** — no 8B re-generation. **Run notebook 5 with `TARGET='truthfulqa'` first** to produce that file.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from truthfulqa_judge import relabel_and_eval
# ---- CONFIG --------------------------------------------------------------------------
JUDGE = 'nli'        # 'nli' (strong entailment model, ~1GB) or 'bleurtacc' (free, BLEURT env)
N     = None         # None = all 817; set e.g. 50 for a quick smoke test
# --------------------------------------------------------------------------------------
results, judged = relabel_and_eval(JUDGE, n=N)
results

### Read it
`AUROC (BLEURT-0.5)` is the broken-label number from notebook 5; `AUROC (judge)` is against the proper label. If the detectors jump from ≈0.5 toward the TriviaQA range, that confirms the collapse was a **labelling artifact**, not classifier overfitting — and gives a real TruthfulQA transfer number. `agreement%` shows how often the BLEURT-0.5 label and the judge disagree.